# PyTorch Tensors — Complete Notes & Code

**Reference Lecture:** [Tensors in PyTorch | Video 2 | CampusX](http://www.youtube.com/watch?v=mDsFsnw3SK4)

## 1. What is a Tensor?
A **tensor** is a specialized multi-dimensional array designed for mathematical and computational efficiency. Just as an array generalizes numbers to 1D and a matrix generalizes arrays to 2D, a tensor generalizes arrays to **$n$ dimensions**.

### Tensor Dimensions in Deep Learning
The **dimension** (or rank) of a tensor indicates the number of directions/axes along which it spans:

| Dimension | Structure | Deep Learning Example | Example Shape |
| :--- | :--- | :--- | :--- |
| **0D** | Scalar | Single loss value computed after a forward pass | `()` |
| **1D** | Vector | Word embedding vector in NLP (e.g., Word2Vec, GloVe) | `(300,)` |
| **2D** | Matrix | Grayscale image grid (e.g., MNIST digit) | `(28, 28)` |
| **3D** | 3D Tensor | Colored RGB image (`Height x Width x Channels`) | `(256, 256, 3)` |
| **4D** | 4D Tensor | Batch of RGB images (`Batch x Height x Width x Channels`) | `(32, 128, 128, 3)` |
| **5D** | 5D Tensor | Batch of video clips (`Batch x Frames x Height x Width x Channels`) | `(10, 16, 64, 64, 3)` |

### Why Tensors Are Essential in Deep Learning
1. **Mathematical Efficiency:** Optimized for core neural network computations such as matrix multiplication, addition, and dot products.
2. **Universal Data Representation:** Maps all major real-world data modalities (tabular, text, audio, images, video) into numerical structures.
3. **Hardware Acceleration (GPU/TPU):** Enables massive parallelization across thousands of CUDA cores instead of sequential CPU execution.

### Where Tensors Are Used
* **Data Storage:** Storing input features and training datasets.
* **Model Parameters:** Storing learnable weights ($W$) and biases ($b$) across layers.
* **Operations & Training:** Executing forward-pass linear transformations ($Wx + b$), activation functions, and backward-pass gradient calculations.

## 2. Environment Setup & Hardware Check
Verify the installed PyTorch version and check whether a CUDA-enabled GPU is available in the runtime.

In [3]:
import torch
print("PyTorch Version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Using CPU.")

PyTorch Version: 2.15.0+cu132
GPU is available!
Using GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 3. Creating a Tensor
PyTorch provides several initialization methods similar to NumPy:
* `torch.empty(shape)`: Allocates memory of the given shape without initializing values (displays leftover garbage values in memory).
* `torch.zeros(shape)` / `torch.ones(shape)`: Initializes all elements to `0` or `1` (useful for initializing biases).
* `torch.rand(shape)`: Initializes random floats uniformly distributed in $[0, 1)$ (useful for weight initialization).
* `torch.manual_seed(seed)`: Fixes the random number generator seed so random tensor initialization is reproducible across runs and machines.
* `torch.tensor(data)`: Creates a tensor directly from Python lists or tuples.
* `torch.arange()`, `torch.linspace()`, `torch.eye()`, `torch.full()`: Utility constructors for ranges, evenly spaced points, identity matrices, and constant-filled tensors.

In [4]:
# 1. Using empty (allocates uninitialized memory)
a = torch.empty(2, 3)
print("Empty tensor:\n", a)
print("Object type:", type(a))

# 2. Using zeros and ones
print("\nZeros:\n", torch.zeros(2, 3))
print("\nOnes:\n", torch.ones(2, 3))

# 3. Using rand (non-deterministic without a seed)
print("\nRandom 1:\n", torch.rand(2, 3))
print("\nRandom 2:\n", torch.rand(2, 3))

Empty tensor:
 tensor([[6.0750e-33, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 4.4842e-44, 0.0000e+00]])
Object type: <class 'torch.Tensor'>

Zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])

Ones:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])

Random 1:
 tensor([[0.6573, 0.7808, 0.3738],
        [0.7029, 0.3365, 0.5103]])

Random 2:
 tensor([[0.5229, 0.3288, 0.1502],
        [0.7009, 0.6337, 0.4533]])


In [5]:
# Using manual_seed for reproducible random generation
torch.manual_seed(100)
print("Seeded Random 1:\n", torch.rand(2, 3))

torch.manual_seed(100)
print("\nSeeded Random 2 (Identical):\n", torch.rand(2, 3))

# Creating a custom tensor from a Python list
custom_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("\nCustom Tensor:\n", custom_tensor)

Seeded Random 1:
 tensor([[0.1117, 0.8158, 0.2626],
        [0.4839, 0.6765, 0.7539]])

Seeded Random 2 (Identical):
 tensor([[0.1117, 0.8158, 0.2626],
        [0.4839, 0.6765, 0.7539]])

Custom Tensor:
 tensor([[1, 2, 3],
        [4, 5, 6]])


In [6]:
# arange (start, end, step)
print("using arange ->", torch.arange(0, 10, 2))

# linspace (start, end, total_points)
print("using linspace ->", torch.linspace(0, 10, 10))

# eye (identity matrix)
print("using eye ->\n", torch.eye(5))

# full (shape, fill_value)
print("using full ->\n", torch.full((3, 3), 5))

using arange -> tensor([0, 2, 4, 6, 8])
using linspace -> tensor([ 0.0000,  1.1111,  2.2222,  3.3333,  4.4444,  5.5556,  6.6667,  7.7778,
         8.8889, 10.0000])
using eye ->
 tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
using full ->
 tensor([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]])


## 4. Tensor Shapes
* Use the `.shape` attribute to inspect the dimensions of any tensor (`torch.Size`).
* Use `*_like` functions (`empty_like`, `zeros_like`, `ones_like`, `rand_like`) to generate a new tensor matching the exact shape of an existing tensor.
* **Note on `rand_like`:** If the source tensor contains integers (e.g., `torch.int64`), `torch.rand_like(x)` will raise an error because `rand` generates floating-point numbers. Explicitly pass `dtype=torch.float32`.

In [7]:
x = torch.tensor([[1, 2, 3], [4, 5, 6]])
print("Tensor x:\n", x)
print("Shape of x:", x.shape)

# Creating tensors with the same shape as x
print("\n empty_like:\n", torch.empty_like(x))
print("\n zeros_like:\n", torch.zeros_like(x))
print("\n ones_like:\n", torch.ones_like(x))

# Explicitly specify float dtype since x is int64
print("\n rand_like:\n", torch.rand_like(x, dtype=torch.float32))

Tensor x:
 tensor([[1, 2, 3],
        [4, 5, 6]])
Shape of x: torch.Size([2, 3])

 empty_like:
 tensor([[0, 0, 0],
        [1, 0, 0]])

 zeros_like:
 tensor([[0, 0, 0],
        [0, 0, 0]])

 ones_like:
 tensor([[1, 1, 1],
        [1, 1, 1]])

 rand_like:
 tensor([[0.2627, 0.0428, 0.2080],
        [0.1180, 0.1217, 0.7356]])


## 5. Tensor Data Types (`dtype`)
* **Check dtype:** `tensor.dtype`
* **Assign dtype at creation:** Pass `dtype=torch.<type>` inside `torch.tensor(...)`.
* **Cast existing tensor:** Use `tensor.to(torch.<type>)`.

| Data Type | PyTorch Dtype | Common Use Case |
| :--- | :--- | :--- |
| **32-bit Floating Point** | `torch.float32` | Standard default for deep learning weights and activations |
| **64-bit Floating Point** | `torch.float64` | High-precision scientific/numerical computing |
| **16-bit Floating Point** | `torch.float16` | Mixed-precision GPU training to reduce VRAM and speed up compute |
| **BFloat16** | `torch.bfloat16` | Mixed-precision training on modern GPUs/TPUs with wider dynamic range |
| **8-bit Floating Point** | `torch.float8` | Experimental ultra-low-precision inference/training |
| **8-bit Signed Integer** | `torch.int8` | Quantized models for low-memory inference |
| **16-bit Signed Integer** | `torch.int16` | Intermediate-precision integer tasks |
| **32-bit Signed Integer** | `torch.int32` | General indexing and integer computations |
| **64-bit Signed Integer** | `torch.int64` | Default integer type in PyTorch; large index arrays |
| **8-bit Unsigned Integer** | `torch.uint8` | Raw image pixel values ($0$ to $255$) |
| **Boolean** | `torch.bool` | Logical masks and conditional filtering |
| **Complex 64 / 128** | `torch.complex64`, `torch.complex128` | Signal processing and Fourier transforms |
| **Quantized Integers** | `torch.qint8`, `torch.quint8` | Quantized tensors for edge/production inference |

In [8]:
# Find existing data type
print("Original dtype of x:", x.dtype)

# Assign data type during creation
int_tensor = torch.tensor([1.0, 2.0, 3.0], dtype=torch.int32)
print("Assigned int32:", int_tensor, int_tensor.dtype)

float_tensor = torch.tensor([1, 2, 3], dtype=torch.float64)
print("Assigned float64:", float_tensor, float_tensor.dtype)

# Convert existing tensor to a new data type using .to()
x_float = x.to(torch.float32)
print("Cast x to float32:\n", x_float, x_float.dtype)

Original dtype of x: torch.int64
Assigned int32: tensor([1, 2, 3], dtype=torch.int32) torch.int32
Assigned float64: tensor([1., 2., 3.], dtype=torch.float64) torch.float64
Cast x to float32:
 tensor([[1., 2., 3.],
        [4., 5., 6.]]) torch.float32


## 6. Mathematical Operations on Tensors

### 6.1 Scalar Operations
Arithmetic between a tensor and a single scalar applies the operation element-by-element across the entire tensor.

In [9]:
x = torch.rand(2, 2)
print("Original x:\n", x)

print("\n Addition (+ 2):\n", x + 2)
print("\n Subtraction (- 2):\n", x - 2)
print("\n Multiplication (* 3):\n", x * 3)
print("\n Division (/ 3):\n", x / 3)
print("\n Integer Division ((x * 100) // 3):\n", (x * 100) // 3)
print("\n Modulus (((x * 100) // 3) % 2):\n", ((x * 100) // 3) % 2)
print("\n Power (x ** 2):\n", x ** 2)

Original x:
 tensor([[0.7118, 0.7876],
        [0.4183, 0.9014]])

 Addition (+ 2):
 tensor([[2.7118, 2.7876],
        [2.4183, 2.9014]])

 Subtraction (- 2):
 tensor([[-1.2882, -1.2124],
        [-1.5817, -1.0986]])

 Multiplication (* 3):
 tensor([[2.1353, 2.3627],
        [1.2549, 2.7042]])

 Division (/ 3):
 tensor([[0.2373, 0.2625],
        [0.1394, 0.3005]])

 Integer Division ((x * 100) // 3):
 tensor([[23., 26.],
        [13., 30.]])

 Modulus (((x * 100) // 3) % 2):
 tensor([[1., 0.],
        [1., 0.]])

 Power (x ** 2):
 tensor([[0.5066, 0.6203],
        [0.1750, 0.8125]])


### 6.2 Element-Wise Operations
* **Binary operations** (`+`, `-`, `*`, `/`, `**`, `%`) between two tensors of the same shape operate on corresponding positions.
* **Unary operations** (`abs`, `neg`, `round`, `ceil`, `floor`, `clamp`) transform each element independently. `torch.clamp(tensor, min, max)` bounds all values within `[min, max]`.

In [10]:
a = torch.rand(2, 3)
b = torch.rand(2, 3)

print("a + b:\n", a + b)
print("a - b:\n", a - b)
print("a * b:\n", a * b)
print("a / b:\n", a / b)
print("a ** b:\n", a ** b)
print("a % b:\n", a % b)

# Unary sign operations
c = torch.tensor([1, -2, 3, -4])
print("\nabs(c):", torch.abs(c))
print("neg(c):", torch.neg(c))

# Rounding and clamping
d = torch.tensor([1.9, 2.3, 3.7, 4.4])
print("\nround(d):", torch.round(d))
print("ceil(d):", torch.ceil(d))
print("floor(d):", torch.floor(d))
print("clamp(d, 2, 3):", torch.clamp(d, min=2, max=3))

a + b:
 tensor([[1.5526, 1.7335, 0.6679],
        [1.2502, 0.9229, 1.3130]])
a - b:
 tensor([[ 0.4411, -0.2205, -0.2201],
        [-0.6455, -0.5661,  0.3346]])
a * b:
 tensor([[0.5540, 0.7391, 0.0994],
        [0.2866, 0.1328, 0.4030]])
a / b:
 tensor([[1.7938, 0.7743, 0.5042],
        [0.3190, 0.2397, 1.6841]])
a ** b:
 tensor([[0.9983, 0.7614, 0.5145],
        [0.3218, 0.2771, 0.9096]])
a % b:
 tensor([[0.4411, 0.7565, 0.2239],
        [0.3023, 0.1784, 0.3346]])

abs(c): tensor([1, 2, 3, 4])
neg(c): tensor([-1,  2, -3,  4])

round(d): tensor([2., 2., 4., 4.])
ceil(d): tensor([2., 3., 4., 5.])
floor(d): tensor([1., 2., 3., 4.])
clamp(d, 2, 3): tensor([2.0000, 2.3000, 3.0000, 3.0000])


### 6.3 Reduction Operations
Reduction functions collapse a tensor (or a specified axis) into summary statistics:
* `dim=0` reduces **down the rows** (computes column-wise results).
* `dim=1` reduces **across the columns** (computes row-wise results).
* Statistical functions like `mean`, `std`, and `var` require floating-point tensors (`float32`/`float64`).
* `argmax` and `argmin` return the flattened index of the maximum and minimum elements.

In [11]:
e = torch.randint(size=(2, 3), low=0, high=10, dtype=torch.float32)
print("Tensor e:\n", e)

# Sum
print("\nTotal sum:", torch.sum(e))
print("Column-wise sum (dim=0):", torch.sum(e, dim=0))
print("Row-wise sum (dim=1):", torch.sum(e, dim=1))

# Mean & Median
print("\nMean:", torch.mean(e))
print("Column-wise mean (dim=0):", torch.mean(e, dim=0))
print("Median:", torch.median(e))

# Min, Max, Product
print("\nMax:", torch.max(e), "| Min:", torch.min(e))
print("Product of all elements:", torch.prod(e))

# Standard Deviation & Variance
print("Std Dev:", torch.std(e), "| Variance:", torch.var(e))

# Indices of max and min values
print("Argmax index:", torch.argmax(e), "| Argmin index:", torch.argmin(e))

Tensor e:
 tensor([[8., 0., 7.],
        [0., 0., 9.]])

Total sum: tensor(24.)
Column-wise sum (dim=0): tensor([ 8.,  0., 16.])
Row-wise sum (dim=1): tensor([15.,  9.])

Mean: tensor(4.)
Column-wise mean (dim=0): tensor([4., 0., 8.])
Median: tensor(0.)

Max: tensor(9.) | Min: tensor(0.)
Product of all elements: tensor(0.)
Std Dev: tensor(4.4272) | Variance: tensor(19.6000)
Argmax index: tensor(5) | Argmin index: tensor(1)


### 6.4 Matrix & Linear Algebra Operations
* `torch.matmul(f, g)`: Matrix multiplication between compatible shapes `(m, n)` and `(n, p)` yielding `(m, p)`.
* `torch.dot(v1, v2)`: Dot product of two 1D vectors.
* `torch.transpose(tensor, dim0, dim1)`: Swaps two specified dimensions.
* `torch.det(h)` & `torch.inverse(h)`: Determinant and matrix inverse (requires a square floating-point matrix).

In [12]:
f = torch.randint(size=(2, 3), low=0, high=10)
g = torch.randint(size=(3, 2), low=0, high=10)

print("Matrix Multiplication (2x3 @ 3x2):\n", torch.matmul(f, g))

# Dot product of 1D vectors
vector1 = torch.tensor([1, 2])
vector2 = torch.tensor([3, 4])
print("\nDot product:", torch.dot(vector1, vector2))

# Transpose
print("\nTranspose of f (3x2):\n", torch.transpose(f, 0, 1))

# Determinant and Inverse (requires square float matrix)
h = torch.randint(size=(3, 3), low=0, high=10, dtype=torch.float32)
print("\nSquare matrix h:\n", h)
print("Determinant of h:", torch.det(h))
print("Inverse of h:\n", torch.inverse(h))

Matrix Multiplication (2x3 @ 3x2):
 tensor([[ 87, 119],
        [ 65,  99]])

Dot product: tensor(11)

Transpose of f (3x2):
 tensor([[5, 9],
        [7, 4],
        [3, 0]])

Square matrix h:
 tensor([[5., 9., 8.],
        [9., 7., 9.],
        [2., 6., 7.]])
Determinant of h: tensor(-110.)
Inverse of h:
 tensor([[ 0.0455,  0.1364, -0.2273],
        [ 0.4091, -0.1727, -0.2455],
        [-0.3636,  0.1091,  0.4182]])


### 6.5 Comparison Operations & Special Functions
* **Comparison operators** (`>`, `<`, `==`, `!=`, `>=`, `<=`) return boolean tensors (`torch.bool`).
* **Special / Activation functions:** `log`, `exp`, `sqrt`, `sigmoid`, `softmax`, and `relu` are built directly into `torch`. Note that `softmax` requires a floating-point tensor and a target dimension `dim`.

In [13]:
i = torch.randint(size=(2, 3), low=0, high=10)
j = torch.randint(size=(2, 3), low=0, high=10)

print("i > j:\n", i > j)
print("i < j:\n", i < j)
print("i == j:\n", i == j)
print("i != j:\n", i != j)
print("i >= j:\n", i >= j)
print("i <= j:\n", i <= j)

# Special functions & Neural Network Activations
k = torch.randint(size=(2, 3), low=1, high=10, dtype=torch.float32)
print("\nTensor k:\n", k)
print("log(k):\n", torch.log(k))
print("exp(k):\n", torch.exp(k))
print("sqrt(k):\n", torch.sqrt(k))
print("sigmoid(k):\n", torch.sigmoid(k))
print("softmax(k, dim=0):\n", torch.softmax(k, dim=0))
print("relu(k):\n", torch.relu(k))

i > j:
 tensor([[ True,  True, False],
        [ True, False, False]])
i < j:
 tensor([[False, False,  True],
        [False,  True,  True]])
i == j:
 tensor([[False, False, False],
        [False, False, False]])
i != j:
 tensor([[True, True, True],
        [True, True, True]])
i >= j:
 tensor([[ True,  True, False],
        [ True, False, False]])
i <= j:
 tensor([[False, False,  True],
        [False,  True,  True]])

Tensor k:
 tensor([[6., 5., 1.],
        [1., 3., 1.]])
log(k):
 tensor([[1.7918, 1.6094, 0.0000],
        [0.0000, 1.0986, 0.0000]])
exp(k):
 tensor([[403.4288, 148.4132,   2.7183],
        [  2.7183,  20.0855,   2.7183]])
sqrt(k):
 tensor([[2.4495, 2.2361, 1.0000],
        [1.0000, 1.7321, 1.0000]])
sigmoid(k):
 tensor([[0.9975, 0.9933, 0.7311],
        [0.7311, 0.9526, 0.7311]])
softmax(k, dim=0):
 tensor([[0.9933, 0.8808, 0.5000],
        [0.0067, 0.1192, 0.5000]])
relu(k):
 tensor([[6., 5., 1.],
        [1., 3., 1.]])


## 7. In-Place Operations
Standard tensor operations (e.g., `m + n` or `torch.relu(m)`) allocate a **new tensor in memory** to store the result. When working with large datasets or deep networks, this increases memory overhead.

* Any PyTorch method ending with a **trailing underscore (`_`)** modifies the tensor **in-place** without allocating new memory (e.g., `m.add_(n)`, `m.relu_()`).

In [14]:
m = torch.rand(2, 3)
n = torch.rand(2, 3)

print("Original m:\n", m)
print("Original n:\n", n)

# In-place addition: updates m directly
m.add_(n)
print("\nm after m.add_(n):\n", m)

# Standard relu (creates a new tensor, leaves m unchanged)
_ = torch.relu(m)

# In-place relu (modifies m directly)
m.relu_()
print("\nm after m.relu_():\n", m)

Original m:
 tensor([[0.6574, 0.3451, 0.0453],
        [0.9798, 0.5548, 0.6868]])
Original n:
 tensor([[0.4920, 0.0748, 0.9605],
        [0.3271, 0.0103, 0.9516]])

m after m.add_(n):
 tensor([[1.1494, 0.4199, 1.0058],
        [1.3069, 0.5650, 1.6384]])

m after m.relu_():
 tensor([[1.1494, 0.4199, 1.0058],
        [1.3069, 0.5650, 1.6384]])


## 8. Copying a Tensor
* **Assignment (`b = a`):** Creates a reference pointing to the **exact same memory address** (`id(a) == id(b)`). Modifying `a` mutates `b`.
* **Deep Copy (`b = a.clone()`):** Allocates a **separate memory location** (`id(a) != id(b)`). Modifying `a` leaves `b` unaffected.

In [15]:
a = torch.rand(2, 3)

# Reference assignment (shares memory)
b = a
a[0][0] = 0
print("After mutating a[0][0] = 0:")
print("a:\n", a)
print("b (also changed!):\n", b)
print("Same memory ID?", id(a) == id(b), f"({id(a)} vs {id(b)})")

# Deep copy using .clone() (independent memory)
b = a.clone()
a[0][0] = 10
print("\nAfter cloning and mutating a[0][0] = 10:")
print("a:\n", a)
print("b (preserved!):\n", b)
print("Same memory ID?", id(a) == id(b), f"({id(a)} vs {id(b)})")

After mutating a[0][0] = 0:
a:
 tensor([[0.0000, 0.2324, 0.9141],
        [0.7668, 0.1659, 0.4393]])
b (also changed!):
 tensor([[0.0000, 0.2324, 0.9141],
        [0.7668, 0.1659, 0.4393]])
Same memory ID? True (127884772496464 vs 127884772496464)

After cloning and mutating a[0][0] = 10:
a:
 tensor([[10.0000,  0.2324,  0.9141],
        [ 0.7668,  0.1659,  0.4393]])
b (preserved!):
 tensor([[0.0000, 0.2324, 0.9141],
        [0.7668, 0.1659, 0.4393]])
Same memory ID? False (127884772496464 vs 127884772497344)


## 9. Running Tensor Operations on GPU
By default, PyTorch creates tensors in CPU RAM. To leverage GPU VRAM and CUDA cores:
1. Define a device object: `device = torch.device('cuda')`
2. Either create tensors directly on the GPU (`torch.rand((2, 3), device=device)`) or transfer an existing CPU tensor using `.to(device)`.
3. Both operands in a calculation must reside on the **same device**.

In [16]:
import time

if torch.cuda.is_available():
    device = torch.device("cuda")

    # Method 1: Create directly on GPU
    gpu_tensor = torch.rand((2, 3), device=device)
    print("Created on GPU:", gpu_tensor.device)

    # Method 2: Move existing CPU tensor to GPU
    cpu_tensor = torch.rand((2, 3))
    moved_tensor = cpu_tensor.to(device)
    print("Moved to GPU:", moved_tensor.device)

    # Benchmark: 10,000 x 10,000 Matrix Multiplication (CPU vs GPU)
    size = 10000
    mat_cpu1 = torch.randn(size, size)
    mat_cpu2 = torch.randn(size, size)

    start_cpu = time.time()
    res_cpu = torch.matmul(mat_cpu1, mat_cpu2)
    cpu_time = time.time() - start_cpu

    mat_gpu1 = mat_cpu1.to(device)
    mat_gpu2 = mat_cpu2.to(device)
    torch.cuda.synchronize()  # ensure accurate GPU timing

    start_gpu = time.time()
    res_gpu = torch.matmul(mat_gpu1, mat_gpu2)
    torch.cuda.synchronize()
    gpu_time = time.time() - start_gpu

    print(f"\nCPU Time: {cpu_time:.4f}s")
    print(f"GPU Time: {gpu_time:.4f}s")
    print(f"Speedup:  {cpu_time / gpu_time:.1f}x faster on GPU")
else:
    print("Enable a GPU runtime (e.g., T4 GPU in Colab) to run the GPU benchmark.")

Created on GPU: cuda:0
Moved to GPU: cuda:0

CPU Time: 3.6150s
GPU Time: 0.5249s
Speedup:  6.9x faster on GPU


## 10. Reshaping Tensors
Deep learning architectures frequently require changing tensor dimensions without altering underlying data:
* `reshape(new_shape)`: Changes shape while preserving total element count.
* `flatten()`: Collapses a multi-dimensional tensor into a 1D vector.
* `permute(dims)`: Reorders all dimensions according to a specified tuple of axis indices.
* `unsqueeze(dim)`: Inserts a new dimension of size `1` at `dim` (commonly used to convert a single image `(H, W, C)` into a batch of size 1 `(1, H, W, C)`).
* `squeeze(dim)`: Removes dimensions of size `1`.

In [17]:
a = torch.ones(4, 4)
print("Original shape:", a.shape)

# reshape (total elements must remain 16)
print("Reshaped (2, 2, 2, 2):", a.reshape(2, 2, 2, 2).shape)

# flatten into 1D
print("Flattened shape:", a.flatten().shape)

# permute axes
b = torch.rand(2, 3, 4)
print("\nOriginal b shape:", b.shape)
print("Permuted b shape (2, 0, 1):", b.permute(2, 0, 1).shape)

# unsqueeze: add batch dimension at index 0 for a single RGB image
image = torch.rand(226, 226, 3)
batched_image = image.unsqueeze(0)
print("\nSingle image shape:", image.shape)
print("After unsqueeze(0):", batched_image.shape)

# squeeze: remove singleton dimension of size 1
d = torch.rand(1, 20)
print("\nBefore squeeze:", d.shape)
print("After squeeze(0):", d.squeeze(0).shape)

Original shape: torch.Size([4, 4])
Reshaped (2, 2, 2, 2): torch.Size([2, 2, 2, 2])
Flattened shape: torch.Size([16])

Original b shape: torch.Size([2, 3, 4])
Permuted b shape (2, 0, 1): torch.Size([4, 2, 3])

Single image shape: torch.Size([226, 226, 3])
After unsqueeze(0): torch.Size([1, 226, 226, 3])

Before squeeze: torch.Size([1, 20])
After squeeze(0): torch.Size([20])


## 11. Converting Between NumPy Arrays & PyTorch Tensors
* **PyTorch Tensor $\rightarrow$ NumPy Array:** `tensor.numpy()`
* **NumPy Array $\rightarrow$ PyTorch Tensor:** `torch.from_numpy(ndarray)`
* *Note:* On CPU, both objects share underlying memory, and `.numpy()` requires the tensor to be on CPU first.

In [18]:
import numpy as np

# PyTorch Tensor -> NumPy Array
t = torch.tensor([1, 2, 3])
arr = t.numpy()
print("Converted to NumPy:", arr, "| Type:", type(arr))

# NumPy Array -> PyTorch Tensor
np_arr = np.array([1, 2, 3])
t_from_np = torch.from_numpy(np_arr)
print("Converted to PyTorch:", t_from_np, "| Type:", type(t_from_np))

Converted to NumPy: [1 2 3] | Type: <class 'numpy.ndarray'>
Converted to PyTorch: tensor([1, 2, 3]) | Type: <class 'torch.Tensor'>
